# Notebook 10 — Combining Datasets

## Learning objectives
Combine tables vertically or horizontally and understand join keys and deprecated append patterns.

**How to use this notebook:** Read each Markdown cell, run the following Python cell, compare your output with the explanation, and then try changing the sample values. The examples use small, fictional datasets so they can run locally without downloading data.

## Setup
NumPy and Pandas are common Python libraries for numerical computation and tabular data analysis. If they are not installed, run `pip install numpy pandas openpyxl pyarrow` in your terminal or notebook environment.

In [1]:
import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 2.3.5
Pandas version: 2.2.3


## Sample tables

In [2]:
customers = pd.DataFrame({'customer_id': [1, 2, 3], 'name': ['Ava', 'Ben', 'Chen']})
orders = pd.DataFrame({'order_id': [101, 102, 103], 'customer_id': [1, 2, 2], 'amount': [50, 75, 30]})
jan = pd.DataFrame({'month': ['Jan'], 'sales': [100]})
feb = pd.DataFrame({'month': ['Feb'], 'sales': [120]})
print(customers)
print(orders)

   customer_id  name
0            1   Ava
1            2   Ben
2            3  Chen
   order_id  customer_id  amount
0       101            1      50
1       102            2      75
2       103            2      30


## concat()

### 1. Concept explanation
`concat` stacks objects along rows or columns; it aligns columns by label.

### 2. Real-world analogy
Combine monthly report pages.

### 3. Business example
Append rows from multiple monthly files.

### 4. AI/ML example
Join compatible batches of observations.

In [3]:
print(pd.concat([jan, feb], ignore_index=True))

  month  sales
0   Jan    100
1   Feb    120


### 5. Output explanation
A two-row table contains January and February sales.

### 6. Best practices
Validate matching column meanings before concatenating.

### 7. Common mistakes
Different schemas may create missing values.

### 8. Advantages
Simple for stacking datasets.

### 9. Limitations
Does not match related records by key.

## merge()

### 1. Concept explanation
`merge` matches rows using one or more key columns, similar to a database join.

### 2. Real-world analogy
Match each order with the customer who placed it.

### 3. Business example
Combine customer details with transaction records.

### 4. AI/ML example
Attach labels or reference data to features.

In [4]:
result = pd.merge(orders, customers, on='customer_id', how='left', validate='many_to_one')
print(result)

   order_id  customer_id  amount name
0       101            1      50  Ava
1       102            2      75  Ben
2       103            2      30  Ben


### 5. Output explanation
Each order is enriched with the matching customer name.

### 6. Best practices
Choose `how` explicitly and use `validate` when cardinality is known.

### 7. Common mistakes
Duplicate keys can multiply rows unexpectedly.

### 8. Advantages
Powerful relational joining.

### 9. Limitations
Bad keys can cause missing matches or row explosion.

## join()

### 1. Concept explanation
`join` is convenient for joining on indexes by default.

### 2. Real-world analogy
Match records using their assigned reference numbers.

### 3. Business example
Attach a lookup table indexed by customer ID.

### 4. AI/ML example
Join pre-indexed feature and metadata tables.

In [5]:
customer_indexed = customers.set_index('customer_id')
orders_indexed = orders.set_index('customer_id')
print(orders_indexed.join(customer_indexed, how='left'))

             order_id  amount name
customer_id                       
1                 101      50  Ava
2                 102      75  Ben
2                 103      30  Ben


### 5. Output explanation
The customer name is joined to each order using the index.

### 6. Best practices
Make index alignment explicit and reset indexes if needed.

### 7. Common mistakes
Joining on the wrong index can silently misalign rows.

### 8. Advantages
Concise for index-based joins.

### 9. Limitations
Less intuitive when keys are ordinary columns.

## append() deprecation and alternatives

### 1. Concept explanation
`DataFrame.append` was deprecated and removed in modern Pandas. Use `pd.concat` instead.

### 2. Real-world analogy
Use the current approved way to add pages to a report.

### 3. Business example
Stack incremental data extracts.

### 4. AI/ML example
Combine batches from a data pipeline.

In [6]:
combined = pd.concat([jan, feb], ignore_index=True)
print(combined)

  month  sales
0   Jan    100
1   Feb    120


### 5. Output explanation
The two frames are combined without using the removed `append` method.

### 6. Best practices
Use one concat for a list of frames rather than repeated concatenation in a loop.

### 7. Common mistakes
Old tutorials may show `.append()` that no longer works.

### 8. Advantages
Modern supported pattern.

### 9. Limitations
`concat` still requires compatible schemas.

## combine_first()

### 1. Concept explanation
`combine_first` fills missing values in one aligned object from another object.

### 2. Real-world analogy
Fill blanks in a form using a second copy of the same form.

### 3. Business example
Combine two partial customer profiles.

### 4. AI/ML example
Merge feature values from alternate sources where one is missing.

In [7]:
left = pd.DataFrame({'email': ['a@example.com', None], 'phone': [None, '555-0100']}, index=[1, 2])
right = pd.DataFrame({'email': ['backup@example.com', 'b@example.com'], 'phone': ['555-0001', '555-0002']}, index=[1, 2])
print(left.combine_first(right))

           email     phone
1  a@example.com  555-0001
2  b@example.com  555-0100


### 5. Output explanation
Missing cells in the left table are filled from the right table using matching indexes/columns.

### 6. Best practices
Check index alignment before combining.

### 7. Common mistakes
It fills missing values; it does not resolve conflicting nonmissing values.

### 8. Advantages
Useful for complementary sources.

### 9. Limitations
Conflicts require a separate business rule.